# 20 — Contextual Embeddings & BERT Mechanics

**Learning objective.** Demonstrate why the same token can receive different contextual vectors and connect the result to masked-language-model pretraining.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


BERT-style encoders are **bidirectional contextual encoders**. A token representation depends on surrounding tokens, unlike static Word2Vec vectors. Pretraining commonly uses masked-token prediction; downstream tasks fine-tune the encoder or consume its embeddings.

In [2]:
import torch, torch.nn as nn
torch.manual_seed(7)
vocab={'[PAD]':0,'bank':1,'river':2,'money':3,'near':4,'holds':5}
emb=nn.Embedding(len(vocab),12)
layer=nn.TransformerEncoderLayer(d_model=12,nhead=3,dim_feedforward=24,batch_first=True,dropout=0.0)
enc=nn.TransformerEncoder(layer,1)
def encode(words):
    ids=torch.tensor([[vocab[w] for w in words]])
    pos=torch.arange(ids.shape[1]).float().view(1,-1,1)
    # deterministic sinusoid-like scalar broadcast just to make position visible
    x=emb(ids)+0.01*pos
    return enc(x).detach()[0]
a=['bank','holds','money']; b=['bank','near','river']
ea,eb=encode(a),encode(b)
cos=nn.functional.cosine_similarity(ea[0],eb[0],dim=0).item()
print('same lexical token:',a[0],b[0])
print('cosine between contextual bank vectors:',round(cos,4))
print('vectors identical?',torch.allclose(ea[0],eb[0]))

same lexical token: bank bank
cosine between contextual bank vectors: 0.963
vectors identical? False


/tmp/ipykernel_1187/380871306.py:6: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  enc=nn.TransformerEncoder(layer,1)


This is a **mechanics demonstration**, not a pretrained BERT checkpoint. It deliberately uses PyTorch's standard transformer encoder so the contextualization effect is observable offline. In production, `transformers.AutoTokenizer` and `AutoModel` load a pretrained checkpoint whose parameters were learned from large corpora.

---
    ## Production takeaways
    - Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
    - Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
    - Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Distinguish static and contextual token embeddings
- Explain masked-language-model pretraining at a systems level